In [1]:
import pandas as pd
import os 
import sys
sys.path.append('../src')

from preprocessing import get_dfs, create_static_df, create_notes_df

In [2]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)

In [3]:
notes = create_notes_df(dfs)
notes

Reading notes from exams.csv
Found 206509 texts
Loading 215137 texts from clinical assessments
Concatenated texts and deleted NaNs, final count: 367140
Average texts per patient: 106.9


KeyboardInterrupt: 

In [ ]:
print(dfs['exams'].columns)
print(dfs['exams'].count())

Index(['PatientID', 'TransplantationID', 'SpenderID', 'DateofTransplantation',
       'Datum', 'Art', 'Organ', 'Befund', 'X', 'Y'],
      dtype='object')
PatientID                206509
TransplantationID        206509
SpenderID                204465
DateofTransplantation    206509
Datum                    206509
Art                      206257
Organ                     39760
Befund                   205959
X                        206509
Y                         35519
dtype: int64


In [ ]:
print("Reading notes from exams.csv")
notes = dfs['exams'][['PatientID', 'TransplantationID', 'X', 'Befund', 'Art']].rename(
        columns={
            'PatientID': 'patient_id',
            'TransplantationID': 'transplant_id',
            'X': 'rel_days', 
            'Befund': 'text',
            'Art': 'type',
        }
    )
print(f"Found {len(notes)} texts")
notes

Reading notes from exams.csv
Found 206509 texts


,patient_id,transplant_id,rel_days,text,type
0,33,1805,"-1,00",Thorax bed side vom 18.12.2003 19:45 : <br> <b...,Röntgen
1,33,1805,"0,00",<br> <br> Farbkodierte Dopplersonographie und ...,Sono
2,33,1805,"1,00",Farbkodierte Dopplersonographie und Power-Dopp...,Sono
3,33,1805,"2,00",Farbkodierte Dopplersonographie und Power-Dopp...,Sono
4,33,1805,"3,00",Farbkodierte Dopplersonographie und Power-Dopp...,Sono
...,...,...,...,...,...
206504,38658,15328,"157,00",==============================================...,Mikrobiologie
206505,38658,15328,"157,00",==============================================...,Mikrobiologie
206506,38658,15328,"157,00",==============================================...,Mikrobiologie
206507,38658,15328,"1224,00",==============================================...,Mikrobiologie


In [ ]:
print(dfs['clinical_assessment'].columns)
print(dfs['clinical_assessment'].count())

Index(['VerlaufID', 'PatientID', 'TransplantationID', 'DateofTransplantation',
       'Datum', 'Date of graft loss', 'OPDtime', 'HatBeurteilung',
       'Beurteilung', 'HatBeurteilungAerztlich', 'BeurteilungAerztlich',
       'HatBeurteilungIntern', 'BeurteilungIntern', 'Blutdruck_systolisch',
       'Blutdruck_diastolisch', 'Gewicht', 'Urinvolumen', 'naechster_Termin',
       'Herzfrequenz', 'Infektion', 'Temperatur', 'Zielwert_Bezeichnung',
       'Zielwert_Wert', 'Zielwert_Einheit', 'Diuresezeit',
       'naechster_TerminZeit', 'Quelle', 'Gesehen_durch', 'Therapie_durch',
       'Uhrzeit', 'naechster_TerminArt', 'OPDGFtime'],
      dtype='object')
VerlaufID                  215137
PatientID                  215137
TransplantationID          215137
DateofTransplantation      215137
Datum                      215137
Date of graft loss          38964
OPDtime                    215137
HatBeurteilung             215137
Beurteilung                 89859
HatBeurteilungAerztlich    215137
B

In [ ]:
# Select and rename the relevant columns from clinical_assessment to match the structure of notes
clinical_assessment_subset = dfs['clinical_assessment'][['PatientID', 'TransplantationID', 'OPDtime', 'BeurteilungAerztlich']].copy()
clinical_assessment_subset.rename(
    columns={
        'PatientID': 'patient_id',
        'TransplantationID': 'transplant_id',
        'OPDtime': 'rel_days',
        'BeurteilungAerztlich': 'text'
    },
    inplace=True
)

clinical_assessment_subset['type'] = 'clinical_assessment'
print(f"Loading {len(clinical_assessment_subset)} texts from clinical assessments")
# Append the transformed clinical_assessment data to notes
notes = pd.concat([notes, clinical_assessment_subset], ignore_index=True)
notes = notes.dropna(subset=['text'])

# remove patients not in static df
notes = notes.merge(
    static_df[['patient_id', 'transplant_id']],
    how='inner',  # Inner join to keep only matching entries
    on=['patient_id', 'transplant_id']
)

print(f"Concatenated texts and deleted NaNs, final count: {len(notes)}")
print(f"Average texts per patient: {len(notes) / len(static_df):.1f}")

Loading 215137 texts from clinical assessments
Concatenated texts and deleted NaNs, final count: 367140
Average texts per patient: 106.9


In [ ]:
# Replace multiple <br> tags with a single whitespace in the notes
notes['text'] = notes['text'].str.replace(r'(<br>\s*)+', ' ', regex=True)

In [ ]:
notes

,patient_id,transplant_id,rel_days,text,type
0,33,1805,"-1,00",Thorax bed side vom 18.12.2003 19:45 : Herz i...,Röntgen
1,33,1805,"0,00",Farbkodierte Dopplersonographie und Power-Dop...,Sono
2,33,1805,"1,00",Farbkodierte Dopplersonographie und Power-Dopp...,Sono
3,33,1805,"2,00",Farbkodierte Dopplersonographie und Power-Dopp...,Sono
4,33,1805,"3,00",Farbkodierte Dopplersonographie und Power-Dopp...,Sono
...,...,...,...,...,...
367135,37446,14926,"194,00",AZ stabil. Durchfall etwas besser aber noch da...,clinical_assessment
367136,37446,14926,"258,00",AZ gut. Medikation seit letztem Termin unverän...,clinical_assessment
367137,37446,14926,"323,00","CellCept-Reduktion am 20.04. von 2g auf 1,5g/T...",clinical_assessment
367138,37446,14926,"397,00",Durchfall etwas besser.\nCorona: inzwischen 4x...,clinical_assessment


In [ ]:
notes

,patient_id,transplant_id,rel_days,text,type
0,33,1805,"-1,00",Thorax bed side vom 18.12.2003 19:45 : Herz i...,Röntgen
1,33,1805,"0,00",Farbkodierte Dopplersonographie und Power-Dop...,Sono
2,33,1805,"1,00",Farbkodierte Dopplersonographie und Power-Dopp...,Sono
3,33,1805,"2,00",Farbkodierte Dopplersonographie und Power-Dopp...,Sono
4,33,1805,"3,00",Farbkodierte Dopplersonographie und Power-Dopp...,Sono
...,...,...,...,...,...
367135,37446,14926,"194,00",AZ stabil. Durchfall etwas besser aber noch da...,clinical_assessment
367136,37446,14926,"258,00",AZ gut. Medikation seit letztem Termin unverän...,clinical_assessment
367137,37446,14926,"323,00","CellCept-Reduktion am 20.04. von 2g auf 1,5g/T...",clinical_assessment
367138,37446,14926,"397,00",Durchfall etwas besser.\nCorona: inzwischen 4x...,clinical_assessment
